In [1]:
%loadlibs
from scipy.integrate import solve_ivp
import pandas as pd
device = 'mps'
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)

Loaded libraries:
	- numpy (np)
	- matplotlib.pyplot (plt)
	- torch
	- torch.nn (nn)
	- torch.optim (optim)
	- tqdm


In [5]:
n_epochs = 5000
n_points = 2000
n_experiments = 1

In [6]:
l = 1.0
g = 9.81
T_MAX = 3.0
n_t = 512

def pendule(t, y, g, l):
    theta, theta_dot = y
    dtheta_dt = theta_dot
    dtheta_dot_dt = -(g / l) * np.sin(theta)
    return [dtheta_dt, dtheta_dot_dt]

res = {'y0': [], 't': [], 'y': []}
y_dot0 = 0.0
t_span = (0, T_MAX)
t_eval = np.linspace(*t_span, n_t)
for y0 in np.linspace(-np.pi, np.pi, 361):
    y, y_dot = solve_ivp(pendule, t_span, [y0, y_dot0], args=(g, l), t_eval=t_eval).y
    res['y0'].extend(np.ones(n_t) * y0)
    res['t'].extend(t_eval)
    res['y'].extend(y)
res['y0'] = np.array(res['y0'])
res['t'] = np.array(res['t'])
res['y'] = np.array(res['y'])
df_pendulum = pd.DataFrame(res)

In [10]:
class PendulumPINN(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.l1 = nn.Linear(2, hidden)
        self.layers = nn.ModuleList([nn.Tanh(),
                                     nn.Linear(hidden, hidden), 
                                     nn.Tanh(),
                                     nn.Linear(hidden, hidden), 
                                     nn.Tanh(),
                                     nn.Linear(hidden, hidden), 
                                     nn.Tanh(),
                                     nn.Linear(hidden, hidden), 
                                     nn.Tanh()])
        self.l3 = nn.Linear(hidden, 1)
        
    def forward(self, y0, t):
        inp = torch.cat([y0, t], dim=1)
        h = self.l1(inp)
        for layer in self.layers:
            h = layer(h)
        nn_out = self.l3(h)
        return (nn_out * t**2 + y0)

def loss_fn(model, y0, t):
    t = t.clone().requires_grad_(True)
    theta = model(y0, t)
    theta_t = torch.autograd.grad(theta, t, grad_outputs=torch.ones_like(theta), create_graph=True, retain_graph=True)[0]
    theta_tt = torch.autograd.grad(theta_t, t, grad_outputs=torch.ones_like(theta_t), create_graph=True, retain_graph=True)[0]
    residual = theta_tt + (g / l) * torch.sin(theta)
    physics_loss = torch.mean(residual ** 2)

    half_theta_t_squared = 0.5 * theta_t ** 2
    half_theta_t_squared_t = torch.autograd.grad(half_theta_t_squared, t, grad_outputs=torch.ones_like(half_theta_t_squared), create_graph=True, retain_graph=True)[0]
    residual = half_theta_t_squared_t - theta_t * (-(g / l) * torch.sin(theta))
    physics_loss2 = torch.mean(residual ** 2)
    return physics_loss, physics_loss2

torch.random.manual_seed(0)
errors_pendulum = np.zeros(len(df_pendulum['y0'].unique()))
models_pendulum = [PendulumPINN() for n in range(n_experiments)]

for n in tqdm(range(n_experiments)):
    model = models_pendulum[n]
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=200, min_lr=1e-5)
    
    for _ in (pbar := tqdm(range(n_epochs))):
        optimizer.zero_grad()
        t = torch.rand((n_points, 1)) * T_MAX
        y0 = (torch.rand((n_points, 1))-0.5) * 4 * torch.pi
        loss_physics, loss_energy = loss_fn(model, y0, t)
        pbar.set_description(f"Loss:{loss_physics.item():.4e}")
        loss_physics.backward()
        optimizer.step()
        scheduler.step(loss_physics.item())
    
    for i in range(len(errors_pendulum)):
        y0 = df_pendulum['y0'].unique()[i]
        simulation = df_pendulum[df_pendulum['y0'] == y0]['y'].to_numpy()
        y_hat = model(torch.ones(n_t, 1) * y0, torch.tensor(t_eval, dtype=torch.float32).unsqueeze(-1)).cpu().squeeze().detach().numpy()
        errors_pendulum[i] += np.mean((y_hat-simulation)**2)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]